# 2_2_03 — Silver: lead activities attendees child table

Explodes the `attendees` array of `silver.lead_activites_processed` into `silver.lead_activites_processed_attendees` — one row per meeting attendee with `contact_id`, real `email`, `is_organizer`, `name`, `status`, `user_id`, plus the activity's `lead_id` and `activity_at`.

Identity spine role (verified Nov 1 2026): `attendees.contact_id` = `contacts.id` for 99.97% of attendees, and `attendees.email` is the only real client email anywhere in the CRM export — it is the bridge to Calendly `INVITEE_EMAIL` (also real). This is the crown jewel table for meeting evidence: 410 of 1,630 coaching clients attribute future meetings through it (93% of all resolvable clients).

In [0]:
%run ../utils

In [0]:
# 2_2_03: attendees child table — one row per attendee, keyed to the activity's lead_id
# Option B CDC (parent-driven, engine utils):
#   - watermark on THIS child (bronze_updated_at is carried down from the parent)
#   - read only parents updated after the watermark, LATEST version per activity_id
#     (silver keeps CDC versions; the child represents CURRENT state, like every consumer dedupes)
#   - exact DELETE of all child rows for affected activity_ids (handles changed attendee
#     lists — a timestamp-only delete would leave stale siblings), then engine write_silver
#   - first run (or legacy schema without bronze_updated_at) = full rebuild
from pyspark.sql.functions import col, explode_outer, row_number
from pyspark.sql.window import Window

CHILD = 'lead_activites_processed_attendees'
try:
    watermark = get_silver_watermark(CHILD)
except Exception:
    watermark = None  # legacy schema without bronze_updated_at → full rebuild

src = spark.table('crm_ingestion.silver.lead_activites_processed').select(
    'id', 'lead_id', 'activity_at', 'type', 'bronze_insert_date', 'bronze_updated_at', 'attendees')
if watermark is not None:
    wm_str = watermark.strftime('%Y-%m-%d %H:%M:%S.%f')
    print(f"CDC MODE: parents updated after {wm_str}")
    src = src.filter(col('bronze_updated_at') > watermark)
else:
    print("FULL LOAD: all activities, latest version per activity_id")

_version = Window.partitionBy('id').orderBy(col('bronze_updated_at').desc(), col('bronze_insert_date').desc())
df_in = src.withColumn('_v', row_number().over(_version)).filter(col('_v') == 1).drop('_v')

if watermark is None or len(df_in.take(1)) > 0:
    if watermark is not None:
        spark.sql(f"""
            DELETE FROM crm_ingestion.silver.{CHILD}
            WHERE activity_id IN (
                SELECT id FROM crm_ingestion.silver.lead_activites_processed
                WHERE bronze_updated_at > '{wm_str}'
            )""")
        _CDC_WATERMARK = watermark
    df_attendees = (
        df_in.select(
            col('id').alias('activity_id'),
            'lead_id', 'activity_at', 'type', 'bronze_insert_date', 'bronze_updated_at',
            explode_outer('attendees').alias('att')
        )
        .select(
            'activity_id', 'lead_id', 'activity_at', 'type', 'bronze_insert_date', 'bronze_updated_at',
            col('att.contact_id').alias('contact_id'),
            col('att.email').alias('email'),
            col('att.is_organizer').alias('is_organizer'),
            col('att.name').alias('name'),
            col('att.status').alias('status'),
            col('att.user_id').alias('user_id')
        )
        .dropna(subset=['contact_id', 'email', 'name', 'user_id'], how='all')
    )
    write_silver(df_attendees, CHILD)
    _CDC_WATERMARK = None
else:
    print("  No changed activities — nothing to do")